# Semana 14: Despliegue de Modelos (MLOps)
## Notebook Conceptual (NB1) – De Modelo a API Local

**Propósito:** Llevar un modelo a producción: serialización, creación de API, contenerización básica y monitoreo.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Serializar un modelo entrenado usando pickle y joblib.
- Crear una API REST con Flask que exponga el modelo.
- Probar el endpoint con requests desde el notebook.
- Escribir un Dockerfile básico para contenerizar la API.
- Introducir MLflow para registro de experimentos.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y entrenamos un modelo simple (regresión logística con iris) que servirá como ejemplo para el despliegue.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Serialización
import pickle
import joblib

# Para pruebas de API
import requests
import json

# Para ejecutar procesos externos (Flask, Docker)
import subprocess
import time
import os

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Entrenamiento de un Modelo de Ejemplo

Usaremos el dataset Iris y entrenaremos una regresión logística multiclase.

In [ ]:
# Cargamos datos
iris = load_iris()
X, y = iris.data, iris.target

# Nombres de características y clases
feature_names = iris.feature_names
class_names = iris.target_names

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenamos modelo
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo: {accuracy:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=class_names))

---
## 2. Serialización del Modelo

Guardamos el modelo entrenado en disco para poder cargarlo posteriormente en la API.

### 2.1. Usando pickle

In [ ]:
# Guardar con pickle
with open('modelo_iris.pkl', 'wb') as f:
    pickle.dump(model, f)

# Verificar que se guardó correctamente
print("Modelo guardado como 'modelo_iris.pkl'")

# Cargar y probar
with open('modelo_iris.pkl', 'rb') as f:
    loaded_model_pickle = pickle.load(f)

y_pred_loaded = loaded_model_pickle.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)
print(f"Accuracy del modelo cargado con pickle: {accuracy_loaded:.4f}")

### 2.2. Usando joblib (recomendado para scikit-learn)

In [ ]:
# Guardar con joblib
joblib.dump(model, 'modelo_iris_joblib.pkl')

print("Modelo guardado como 'modelo_iris_joblib.pkl'")

# Cargar y probar
loaded_model_joblib = joblib.load('modelo_iris_joblib.pkl')
y_pred_joblib = loaded_model_joblib.predict(X_test)
accuracy_joblib = accuracy_score(y_test, y_pred_joblib)
print(f"Accuracy del modelo cargado con joblib: {accuracy_joblib:.4f}")

---
## 3. Creación de una API con Flask

Flask es un microframework web para Python. Crearemos un script `app.py` que carga el modelo y expone un endpoint `/predict` que recibe JSON con las características y devuelve la predicción.

In [ ]:
# Contenido del archivo app.py
app_content = '''
import numpy as np
import joblib
from flask import Flask, request, jsonify

# Cargar modelo
model = joblib.load('modelo_iris_joblib.pkl')

# Crear app
app = Flask(__name__)

@app.route('/')
def home():
    return "API de predicción Iris. Use /predict con POST."

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Obtener datos del request
        data = request.get_json()
        
        # Convertir a array
        features = np.array(data['features']).reshape(1, -1)
        
        # Predecir
        prediction = model.predict(features)[0]
        probabilities = model.predict_proba(features)[0].tolist()
        
        # Mapear a nombre de clase
        class_names = ['setosa', 'versicolor', 'virginica']
        
        return jsonify({
            'prediction': int(prediction),
            'class_name': class_names[prediction],
            'probabilities': probabilities
        })
    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

# Guardamos el archivo
with open('app.py', 'w') as f:
    f.write(app_content)

print("Archivo app.py creado.")
!ls -la app.py

### 3.1. Archivo de requisitos (requirements.txt)

Creamos un archivo con las dependencias necesarias para la API.

In [ ]:
requirements_content = '''
flask==2.3.2
numpy==1.24.3
scikit-learn==1.3.0
joblib==1.3.1
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Archivo requirements.txt creado.")
!cat requirements.txt

### 3.2. Ejecutar la API (en segundo plano)

En un entorno real, ejecutaríamos la API en una terminal separada. En Colab, podemos simularlo ejecutando en segundo plano.

In [ ]:
# Verificamos que no haya un proceso previo en el puerto 5000
!lsof -i:5000 || echo "Puerto 5000 libre"

# Ejecutamos Flask en segundo plano
# En Colab, necesitamos usar una forma especial
import subprocess
import time

# Iniciamos el proceso
process = subprocess.Popen(['python', 'app.py'], 
                           stdout=subprocess.PIPE, 
                           stderr=subprocess.PIPE,
                           text=True)

# Esperamos a que el servidor inicie
time.sleep(5)
print("Servidor Flask iniciado en segundo plano.")

---
## 4. Prueba del Endpoint con Requests

Probamos la API enviando datos y recibiendo predicciones.

In [ ]:
# Datos de ejemplo (características de una flor Iris)
test_features = [5.1, 3.5, 1.4, 0.2]  # setosa

# Preparamos el payload
payload = {
    'features': test_features
}

# Enviamos petición
try:
    response = requests.post('http://localhost:5000/predict', 
                             json=payload,
                             headers={'Content-Type': 'application/json'})
    
    if response.status_code == 200:
        result = response.json()
        print("Respuesta de la API:")
        print(f"Predicción: {result['prediction']} ({result['class_name']})")
        print(f"Probabilidades: {result['probabilities']}")
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"Error al conectar: {e}")
    print("Asegúrate de que el servidor Flask esté corriendo.")

In [ ]:
# Probamos con más ejemplos del test set
for i in range(3):
    features = X_test[i].tolist()
    true_label = y_test[i]
    
    payload = {'features': features}
    response = requests.post('http://localhost:5000/predict', json=payload)
    
    if response.status_code == 200:
        result = response.json()
        print(f"Ejemplo {i+1}:")
        print(f"  Real: {true_label} ({class_names[true_label]})")
        print(f"  Predicho: {result['prediction']} ({result['class_name']})")
        print(f"  Probabilidades: {[round(p, 3) for p in result['probabilities']]}")
        print()
    else:
        print(f"Error en ejemplo {i+1}")

### 4.1. Detener el servidor Flask

Después de las pruebas, detenemos el proceso.

In [ ]:
process.terminate()
print("Servidor Flask detenido.")

---
## 5. Contenerización con Docker

Docker permite empaquetar la aplicación con todas sus dependencias para ejecutarla de manera reproducible en cualquier sistema.

### 5.1. ¿Qué es Docker?
- Docker es una plataforma de contenedores que permite empaquetar aplicaciones en entornos aislados.
- Un contenedor incluye la aplicación y todas sus dependencias.
- Se define mediante un archivo `Dockerfile`.

### 5.2. Creación del Dockerfile

Creamos un archivo `Dockerfile` que:
- Usa una imagen base de Python.
- Copia los archivos necesarios.
- Instala las dependencias.
- Expone el puerto 5000.
- Define el comando para ejecutar la API.

In [ ]:
dockerfile_content = '''
# Usar imagen oficial de Python
FROM python:3.9-slim

# Establecer directorio de trabajo
WORKDIR /app

# Copiar archivos de requisitos
COPY requirements.txt .

# Instalar dependencias
RUN pip install --no-cache-dir -r requirements.txt

# Copiar modelo y código
COPY modelo_iris_joblib.pkl .
COPY app.py .

# Exponer puerto
EXPOSE 5000

# Comando para ejecutar la aplicación
CMD ["python", "app.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("Archivo Dockerfile creado.")
!cat Dockerfile

### 5.3. Comandos básicos de Docker

Para construir y ejecutar el contenedor, usaríamos los siguientes comandos en una terminal con Docker instalado:

```bash
# Construir la imagen
docker build -t iris-api .

# Ejecutar el contenedor
docker run -p 5000:5000 iris-api

# Ver contenedores en ejecución
docker ps

# Detener contenedor
docker stop <container_id>
```

En Google Colab no podemos ejecutar Docker directamente, pero los comandos son estándar.

In [ ]:
print("Para construir y ejecutar el contenedor Docker:")
print("1. En una terminal con Docker, navega a la carpeta con los archivos.")
print("2. Ejecuta: docker build -t iris-api .")
print("3. Ejecuta: docker run -p 5000:5000 iris-api")
print("4. Prueba la API en http://localhost:5000/predict")

---
## 6. MLflow para Registro de Experimentos

MLflow es una plataforma open-source para gestionar el ciclo de vida del machine learning, incluyendo:
- Registro de experimentos (parámetros, métricas, artefactos).
- Empaquetado de modelos.
- Despliegue.

### 6.1. Instalación de MLflow

In [ ]:
# Instalamos MLflow
!pip install mlflow

### 6.2. Ejemplo de uso de MLflow

In [ ]:
import mlflow
import mlflow.sklearn

# Configurar el tracking URI (local)
mlflow.set_tracking_uri('file:./mlruns')

# Iniciar un experimento
mlflow.set_experiment('Iris Classification')

with mlflow.start_run():
    # Registrar parámetros
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('max_iter', 200)
    mlflow.log_param('solver', 'lbfgs')
    
    # Entrenar modelo
    model = LogisticRegression(max_iter=200, random_state=42)
    model.fit(X_train, y_train)
    
    # Registrar métricas
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric('accuracy', accuracy)
    
    # Guardar modelo
    mlflow.sklearn.log_model(model, 'model')
    
    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")

print("\nExperimento registrado en MLflow.")
print("Para ver la UI, ejecuta en terminal: mlflow ui")

### 6.3. Visualización de experimentos

Para ver la interfaz web de MLflow, ejecuta en una terminal:

```bash
mlflow ui
```

Luego abre http://localhost:5000 en tu navegador.

In [ ]:
# Ejemplo de cómo cargar un modelo guardado por MLflow
runs = mlflow.search_runs(experiment_ids=['1'])
if not runs.empty:
    best_run = runs.loc[runs['metrics.accuracy'].idxmax()]
    run_id = best_run['run_id']
    print(f"Cargando modelo del run {run_id}")
    
    loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
    y_pred = loaded_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy del modelo cargado: {acc:.4f}")
else:
    print("No se encontraron runs.")

---
## 7. Resumen de Archivos Generados

Hemos creado los siguientes archivos para el despliegue:

In [ ]:
!ls -la *.pkl *.py *.txt Dockerfile 2>/dev/null || echo "Archivos listados"

---
## 8. Conclusiones

En este notebook hemos cubierto el flujo completo para llevar un modelo a producción:

✔️ **Serialización**: Guardamos el modelo con pickle y joblib.

✔️ **API REST**: Creamos una API con Flask que expone el modelo.

✔️ **Pruebas**: Verificamos el endpoint con requests.

✔️ **Contenerización**: Escribimos un Dockerfile para empaquetar la aplicación.

✔️ **MLflow**: Introducimos el registro de experimentos y modelos.

**Próximos pasos**:
- Desplegar la API en la nube (AWS, GCP, Azure).
- Implementar monitoreo con herramientas como Evidently.
- Configurar CI/CD para actualizaciones automáticas.

---
**Fin del Notebook Conceptual - Semana 14**